In [13]:
import pandas as pd

import numpy as np

In [3]:
train = pd.read_csv('train_cleaned.csv')

test = pd.read_csv('test_cleaned.csv')

/tmp/ipykernel_2972/453101731.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('train_cleaned.csv')
/tmp/ipykernel_2972/453101731.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv('test_cleaned.csv')


In [9]:
# One row per store, built from train only

stores = train.drop_duplicates('Store')[['Store', 'StoreType', 'Assortment', 'CompetitionDistance']]

print("Stores:", len(stores), "| null distance:", stores['CompetitionDistance'].isna().sum())

Stores: 1115 | null distance: 3


In [10]:
# Segment medians from observed values only

seg_median = stores.groupby(['StoreType', 'Assortment'])['CompetitionDistance'].agg(['median', 'count'])

seg_median

median  count
StoreType Assortment               
a         a           1560.0    381
          c           2320.0    220
b         a           1180.0      7
          b            860.0      9
          c           1210.0      1
c         a           1270.0     77
          c           2100.0     71
d         a           4385.0    126
          c           5345.0    220

In [11]:
med = stores.groupby(['StoreType', 'Assortment'])['CompetitionDistance'].median()

med

StoreType  Assortment
a          a             1560.0
           c             2320.0
b          a             1180.0
           b              860.0
           c             1210.0
c          a             1270.0
           c             2100.0
d          a             4385.0
           c             5345.0
Name: CompetitionDistance, dtype: float64

In [12]:
for df in (train, test):
    df['CompetitionDistance_missing'] = df['CompetitionDistance'].isna().astype(int)
    fill = df.set_index(['StoreType', 'Assortment']).index.map(med)
    df['CompetitionDistance'] = df['CompetitionDistance'].fillna(pd.Series(fill, index=df.index))

for name, df in (('train', train), ('test', test)):
    print(name, "nulls:", df['CompetitionDistance'].isna().sum(),
          "| flagged rows:", df['CompetitionDistance_missing'].sum())
    print(df[df['CompetitionDistance_missing'] == 1].groupby('Store')['CompetitionDistance'].agg(['first', 'count']))

train nulls: 0 | flagged rows: 2642
        first  count
Store               
291    4385.0    942
622    2320.0    942
879    4385.0    758
test nulls: 0 | flagged rows: 96
        first  count
Store               
622    2320.0     48
879    4385.0     48


## 5.1 CompetitionDistance imputation

**What:** Filled the 3 stores with missing CompetitionDistance (291, 622, 879) with their StoreType×Assortment segment median, and added a `CompetitionDistance_missing` flag (1 = filled, 0 = observed).

**Why:** Linear regression and sklearn's random forest cannot take NaN. A global median (~2,325) would understate d/a stores, whose median is 4,385. Dropping the stores would lose 2,642 train rows, and the test set contains stores 622 and 879. A placeholder like 999999 would wrongly claim "no competitor". The flag keeps the filled value visible as a guess.

**How:** Built a one-row-per-store view from train (1,115 stores). Computed segment medians from observed values only: d/a = 4,385 (n=126), a/c = 2,320 (n=220). Applied the same medians to train and test. The flag was created before filling.

**Validation:**
- 0 nulls remain in train and test.
- Train flagged rows = 2,642 (942 + 942 + 758), matching Stage 1.
- Test flagged rows = 96 (stores 622 and 879, 48 days each). Store 291 is not in the test set.

**Limits:** Stores 291 and 879 get the same value although EDA showed they sit at opposite ends of d/a sales. The fill approximates a typical d/a store and does not recover the true distance.

In [14]:
for df in (train, test):
    df['CompetitionDistance_log'] = np.log1p(df['CompetitionDistance'])

for name, df in (('train', train), ('test', test)):
    print(name, "nulls:", df['CompetitionDistance_log'].isna().sum())
    print(df['CompetitionDistance_log'].describe().round(3))

train nulls: 0
count    1017209.000
mean           7.645
std            1.557
min            3.045
25%            6.567
50%            7.754
75%            8.837
max           11.237
Name: CompetitionDistance_log, dtype: float64
test nulls: 0
count    41088.000
mean         7.620
std          1.536
min          3.045
25%          6.581
50%          7.794
75%          8.770
max         11.237
Name: CompetitionDistance_log, dtype: float64


In [15]:
# imputed stores should map to log1p of their fill values

print(np.log1p([4385, 2320]).round(3))

print(train[train['CompetitionDistance_missing'] == 1].groupby('Store')['CompetitionDistance_log'].first().round(3))

[8.386 7.75 ]
Store
291    8.386
622    7.750
879    8.386
Name: CompetitionDistance_log, dtype: float64


## 5.1b CompetitionDistance log transform

**What:** Added `CompetitionDistance_log = log1p(CompetitionDistance)` to train and test. Models will use this column; the raw column stays in the dataframe but is excluded from the model feature list.

**Why:** Raw distance is right-skewed (mean 5,405 vs median 2,325), which would let a few far-away competitors dominate a linear model. log1p is monotonic, so trees are unaffected. One shared column avoids duplicate features and model-specific pipelines.

**How:** Applied np.log1p after the imputation step, so filled values are transformed like observed ones.

**Validation:**
- 0 nulls in train and test.
- Range 3.045–11.237 in both files; test sits inside the train range.
- Imputed stores map correctly: 291 and 879 = 8.386, 622 = 7.750.
- Mean 7.645 and median 7.754 show the skew is largely removed.